1. Scraping list of cat breeds from wikipedia.
   The data includes breed name, location of origin, type, body type, coat type and length, and coat pattern.

In [1]:
!python3 -m pip install requests beautifulsoup4 pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Tanisha\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


2. Import libraries

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

## 3. Store the URL
The URL of the Wikipedia page containing the list of cat breeds is stored in a variable called `url`.

In [6]:
url = "https://en.wikipedia.org/wiki/List_of_cat_breeds"
url

'https://en.wikipedia.org/wiki/List_of_cat_breeds'

4. Request the webpage
The `requests.get()` function sends a request to the webpage and returns a response from the server.

In [8]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response

<Response [200]>

 Get the HTML

The response contains the HTML code of the webpage.


In [9]:
html = response.text
print(html[:1000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-feature-navigation-update-disabled vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>List of cat breeds - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 

In [12]:
soup = BeautifulSoup(html, "html.parser")
type(soup)

bs4.BeautifulSoup

In [13]:
tables = soup.find_all("table")
len(tables)

5

5.Inspect the tables 
The webpage contains five tables

In [14]:
for i, table in enumerate(tables):
    print("TABLE", i)
    print(table.get_text(" ", strip=True)[:500])
    print("-" * 50)

TABLE 0
Breed Image Location of origin Type Body type Coat type and length Coat pattern Abyssinian [ 9 ] Unspecified, but somewhere in Afro-Asia, likely Ethiopia [ 10 ] Natural Semi-foreign Short Ticked tabby Aegean Greece Natural Moderate Semi-long Multi-color American Bobtail [ 11 ] United States [ 12 ] Mutation of shortened tail Cobby Semi-long All American Curl [ 13 ] United States [ 12 ] Mutation Semi-foreign Semi-long All American Shorthair United States [ 12 ] Natural Cobby Short All American Wir
--------------------------------------------------
TABLE 1
v t e Domestic cats Felinology Anatomy Genetics Dwarf cat Kitten Odd-eyed cat Squitten Coat genetics Bicolor cat Black cat Calico cat Maltese cat Tabby cat Tortoiseshell cat Moggy Health Aging Declawing Diet dental health senior Vaccination Behavior Cat–dog relationship Catnip valerian Communication Catfight Meow Purr Kneading Intelligence Play and toys Righting reflex Senses Human–cat interaction Ailurophobia Cat cafés Cat fanc

6. Identifying cat breeds

In [15]:
for i, table in enumerate(tables):
    headers = [cell.get_text(" ", strip=True) for cell in table.find_all("th")]
    print("TABLE", i, ":", headers[:10])

TABLE 0 : ['Breed', 'Image', 'Location of origin', 'Type', 'Body type', 'Coat type and length', 'Coat pattern', 'Abyssinian [ 9 ]', 'Aegean', 'American Bobtail [ 11 ]']
TABLE 1 : ['v t e Domestic cats', 'Felinology', 'Health', 'Behavior', 'Human–cat interaction', 'Registries', 'Breeds ( full list ) ( experimental )', 'Fully domestic', 'Landraces', 'Hybrid']
TABLE 2 : ['Fully domestic', 'Landraces', 'Hybrid']
TABLE 3 : ['Infectious', 'Non-infectious']
TABLE 4 : ['v t e Breeds and cultivars', 'Methods', 'Animal breeds', 'Plant cultivars', 'Selection methods and genetics', 'Other']


In [17]:
breed_table = tables[0]

 7. Inspect the rows

In [18]:
rows = breed_table.find_all("tr")
len(rows)

101

8. Looking at the first few rows

In [19]:
for row in rows[:3]:
    print(row.get_text(" | ", strip=True))
    print("-" * 80)

Breed | Image | Location of origin | Type | Body type | Coat type and length | Coat pattern
--------------------------------------------------------------------------------
Abyssinian | [ | 9 | ] | Unspecified, but somewhere in Afro-Asia, likely | Ethiopia | [ | 10 | ] | Natural | Semi-foreign | Short | Ticked tabby
--------------------------------------------------------------------------------
Aegean | Greece | Natural | Moderate | Semi-long | Multi-color
--------------------------------------------------------------------------------


 8. Extract the data from each row

In [20]:
data = []

for row in rows[1:]:
    cells = row.find_all("td")
    row_data = [cell.get_text(" ", strip=True) for cell in cells]
    data.append(row_data)

data[:3]

[['',
  'Unspecified, but somewhere in Afro-Asia, likely Ethiopia [ 10 ]',
  'Natural',
  'Semi-foreign',
  'Short',
  'Ticked tabby'],
 ['', 'Greece', 'Natural', 'Moderate', 'Semi-long', 'Multi-color'],
 ['',
  'United States [ 12 ]',
  'Mutation of shortened tail',
  'Cobby',
  'Semi-long',
  'All']]

In [21]:
cells = row.find_all("td")

In [22]:
[cell.get_text(" ", strip=True) for cell in cells]

['',
 'New York , United States',
 'Natural',
 'Moderate',
 'Long',
 'Solid chocolate and solid lilac or any of these colours with white']

9. Check the extracted data

In [23]:
for row in data[:3]:
    print(row)

['', 'Unspecified, but somewhere in Afro-Asia, likely Ethiopia [ 10 ]', 'Natural', 'Semi-foreign', 'Short', 'Ticked tabby']
['', 'Greece', 'Natural', 'Moderate', 'Semi-long', 'Multi-color']
['', 'United States [ 12 ]', 'Mutation of shortened tail', 'Cobby', 'Semi-long', 'All']


10. Create a dataframe

Now we convert the list of dictionaries into a Pandas dataframe.

- Each dictionary represents one row (one cat breed).
- Each dictionary key becomes a column header.
- Each value becomes a cell.

In [24]:
cat_df = pd.DataFrame(data)
cat_df.head()

,0,1,2,3,4,5
0,,"Unspecified, but somewhere in Afro-Asia, likel...",Natural,Semi-foreign,Short,Ticked tabby
1,,Greece,Natural,Moderate,Semi-long,Multi-color
2,,United States [ 12 ],Mutation of shortened tail,Cobby,Semi-long,All
3,,United States [ 12 ],Mutation,Semi-foreign,Semi-long,All
4,,United States [ 12 ],Natural,Cobby,Short,All


### Inspect dataframe shape and columns

Check the total number of cat breeds scraped and ensure there are no missing values or incorrect data types.

19. Validate the scrape

In [25]:
cat_df.sample(min(5, len(cat_df)))

,0,1,2,3,4,5
69,,Russia,"Crossbreed between the Donskoy , Oriental Shor...",Oriental,"Hairless, velour, brush, or straight coat",All
71,,United States,Crossbreed between the Ragdoll with limited ou...,Cobby,Long,All
44,,Thailand [ 14 ],Natural,Moderate,Short,Solid white
57,,United States,Crossbreed between the Persian and Munchkin,Dwarf,Short/long,All
6,,Cyprus,Natural,Lean and muscular,All,All


## 21. Save as CSV



In [26]:
cat_df.to_csv("cat_breeds.csv", index=False)
print("Saved cat_breeds.csv successfully!")

Saved cat_breeds.csv successfully!


## 22. Analyse the scraped data


In [27]:
cat_df.describe(include='all')

,0,1,2,3,4,5
count,100,100,100,100,100,100
unique,1,64,57,21,14,41
top,,United States [ 12 ],Natural,Moderate,Short,All
freq,100,15,29,26,50,35


## 23. Select final columns and verify integrity

In [28]:
final_df = cat_df.copy()